<a href="https://colab.research.google.com/github/hilalrusydin95/Hilal/blob/main/hilal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ipywidgets statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

# load data
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/refs/heads/main/dataset_fiskal_97_hipoFull.csv"
df = pd.read_csv(url)
print("Data berhasil dimuat:", df.shape)

kota_list = sorted(df["Kota"].unique())
var_list = df.select_dtypes(include=[np.number]).drop(columns=["Tahun"]).columns.tolist()

# widget
lay = widgets.Layout(width="450px")
sty = {"description_width": "110px"}

dd_kota = widgets.Dropdown(options=kota_list, value="Kota Makassar",
                           description="Kota:", layout=lay, style=sty)
dd_y = widgets.Dropdown(options=var_list, value="DBH_PAJAK",
                        description="Variabel Y:", layout=lay, style=sty)
dd_x1 = widgets.Dropdown(options=var_list, value="bm_pendidikan",
                         description="Variabel X1:", layout=lay, style=sty)
dd_x2 = widgets.Dropdown(options=var_list, value="KETIMPANGAN",
                         description="Variabel X2:", layout=lay, style=sty)
btn = widgets.Button(description="Jalankan Analisis", button_style="success",
                     layout=lay)
out = widgets.Output()


def analisis(b):
    with out:
        clear_output(wait=True)

        kota = dd_kota.value
        y_name = dd_y.value
        x1_name = dd_x1.value
        x2_name = dd_x2.value

        if x1_name == x2_name:
            print("Variabel X1 dan X2 tidak boleh sama")
            return

        d = df[df["Kota"] == kota].dropna(subset=[y_name, x1_name, x2_name])
        if len(d) < 5:
            print("Data", kota, "terlalu sedikit untuk regresi:", len(d), "baris")
            return

        # regresi OLS
        X = sm.add_constant(d[[x1_name, x2_name]])
        model = sm.OLS(d[y_name], X).fit()
        print(model.summary())

        b0 = model.params["const"]
        b1 = model.params[x1_name]
        b2 = model.params[x2_name]
        p1 = model.pvalues[x1_name]
        p2 = model.pvalues[x2_name]
        r2 = model.rsquared

        # interpretasi
        print("\nINTERPRETASI")
        print(f"Persamaan: Y = {b0:.3f} + {b1:.3f} {x1_name} + {b2:.3f} {x2_name}")
        print(f"- Konstanta {b0:.3f}: nilai {y_name} ketika {x1_name} dan {x2_name} sama dengan 0")
        print(f"- Koefisien {x1_name} {b1:.3f}: kenaikan 1 satuan {x1_name} mengubah {y_name} sebesar {b1:.3f} (X2 dianggap tetap)")
        print(f"- Koefisien {x2_name} {b2:.3f}: kenaikan 1 satuan {x2_name} mengubah {y_name} sebesar {b2:.3f} (X1 dianggap tetap)")

        print("\nUji signifikansi (alpha 0.05)")
        for nama, p in [(x1_name, p1), (x2_name, p2)]:
            if p < 0.05:
                print(f"- {nama}: p-value {p:.4f} < 0.05, berpengaruh signifikan terhadap {y_name}")
            else:
                print(f"- {nama}: p-value {p:.4f} > 0.05, tidak berpengaruh signifikan terhadap {y_name}")

        print(f"\nR-squared = {r2:.4f}, artinya {r2*100:.2f}% variasi {y_name} dijelaskan oleh {x1_name} dan {x2_name}, sisanya {(1-r2)*100:.2f}% dijelaskan faktor lain.\n")

        # grafik 3D
        g1 = np.linspace(d[x1_name].min(), d[x1_name].max(), 30)
        g2 = np.linspace(d[x2_name].min(), d[x2_name].max(), 30)
        G1, G2 = np.meshgrid(g1, g2)
        Z = b0 + b1 * G1 + b2 * G2

        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")
        ax.plot_surface(G1, G2, Z, alpha=0.5, cmap="viridis")
        ax.scatter(d[x1_name], d[x2_name], d[y_name], color="blue", s=40, label="Data")
        for i in range(len(d)):
            ax.text(d[x1_name].iloc[i], d[x2_name].iloc[i], d[y_name].iloc[i],
                    str(int(d["Tahun"].iloc[i])), fontsize=8)

        ax.set_xlabel(x1_name)
        ax.set_ylabel(x2_name)
        ax.set_zlabel(y_name)
        ax.set_title(f"Regresi 3D - {kota} (R2 = {r2:.3f})")
        ax.view_init(elev=25, azim=-60)
        ax.legend()
        plt.show()


btn.on_click(analisis)

display(widgets.VBox([dd_kota, dd_y, dd_x1, dd_x2, btn]), out)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.9 MB/s eta 0:00:00
Data berhasil dimuat: (1067, 24)


Output()